# **1. BIBLIOTECAS**

In [ ]:
from datetime import datetime, timedelta, timezone
from IPython.display import display
import pandas as pd
import requests
import urllib3
import json
import jwt
import sys

# **2. DADOS DE ACESSO**

## **2.1. Credenciais**

In [ ]:
CREDENTIALS = {
    "username": "rep\\fabio.demuner",
    "password": "KTw302@*7AvR^mgK",
    "grant_type": "password"
}

## **2.2. URL's**

### **2.2.1. Base**

In [ ]:
BASE_URLS = {
    "ubu": "http://services.repcenter.skf.com:22011",
    "germano": "http://services.repcenter.skf.com:20446"
}

### **2.2.2. Endpoint**

In [ ]:
def build_urls(base_url):
    return {
        "token": f"{base_url}/token",
        "machines": f"{base_url}/v1/machines",
        "parts": f"{base_url}/v1/parts",
        "submachines": f"{base_url}/v1/hierarchy",
        "points": f"{base_url}/v1/points",
        "notes": f"{base_url}/v1/notes"
    }

## **2.3. Funções**

### **2.3.1. Token**

In [ ]:
def obter_token(base_url):
    url = f"{base_url}/token"

    response = requests.post(url, data=CREDENTIALS, verify=False)
    response.raise_for_status()

    return response.json()["access_token"]

### **2.3.2. Hierarquia**

In [ ]:
def obter_arvore_hierarquia(token, base_url):
    url = f"{base_url}/v1/hierarchy"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    return response.json()

## **2.4. Sheets**

### **2.4.1. Autenticação no Sheets**

In [ ]:
import gspread
from google.colab import auth
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth import default

# Autentica no Google
auth.authenticate_user()

# Usar 'default' para obter as credenciais no formato esperado para o gspread
creds, _ = default()
gc = gspread.authorize(creds)

# **3. REQUISIÇÃO: MACHINES**

## **3.1. Execução**

In [ ]:
def get_machines(token, base_url, origem):
    url = f"{base_url}/v1/machines"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    df = pd.DataFrame(response.json())
    df["origem"] = origem

    return df


def get_all_machines():
    tokens = {
        origem: obter_token(base_url)
        for origem, base_url in BASE_URLS.items()
    }

    return pd.concat(
        [
            get_machines(tokens[origem], base_url, origem)
            for origem, base_url in BASE_URLS.items()
        ],
        ignore_index=True
    )


df_machines = get_all_machines()

## **3.2. Estrutura e organização**

In [ ]:
# Garantir string
df_machines["path"] = df_machines["path"].fillna("").astype(str)

# Split
df_path = df_machines["path"].str.split(r"\\", expand=True)
df_path.columns = [f"path_{i}" for i in range(df_path.shape[1])]

# Estrutura
df_machines = pd.concat(
    [df_machines[["origem", "id", "name", "path"]], df_path],
    axis=1
)

# Normalizar
df_machines["path_1"] = df_machines["path_1"].str.strip().str.upper()

# Filtro
mapa = {"ubu": "UBU", "germano": "GERMANO"}
df_machines = df_machines[
    df_machines["path_1"] == df_machines["origem"].map(mapa)
]

# Exibir
display(df_machines.head(100))

,origem,id,name,path,path_0,path_1,path_2,path_3,path_4,path_5,path_6
0,ubu,5,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
1,ubu,50,VENTILADOR - MANCAL LA (RANGE FIXO),SAMARCO\UBU\USINA 4\U04-06VT002\VENTILADOR - M...,SAMARCO,UBU,USINA 4,U04-06VT002,VENTILADOR - MANCAL LA (RANGE FIXO),None,None
2,ubu,51,VENTILADOR - MANCAL LOA (RANGE FIXO),SAMARCO\UBU\USINA 4\U04-06VT002\VENTILADOR - M...,SAMARCO,UBU,USINA 4,U04-06VT002,VENTILADOR - MANCAL LOA (RANGE FIXO),None,None
3,ubu,116,CV16,SAMARCO\UBU\USINA 4\FILTRAGEM\CV00 - BOMBAS VÁ...,SAMARCO,UBU,USINA 4,FILTRAGEM,CV00 - BOMBAS VÁCUO,CV16,None
4,ubu,274,U04-06TP013,SAMARCO\UBU\USINA 4\PENEIRAMENTO\U04-06TP013,SAMARCO,UBU,USINA 4,PENEIRAMENTO,U04-06TP013,None,None
...,...,...,...,...,...,...,...,...,...,...,...
97,germano,10301,G02-03CV004 TA01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...,SAMARCO,GERMANO,G02,BENEFICIAMENTO,TRANSPORTADORES,G02-03CV004,G02-03CV004 TA01
98,germano,10315,G02-03CV004 RD01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...,SAMARCO,GERMANO,G02,BENEFICIAMENTO,TRANSPORTADORES,G02-03CV004,G02-03CV004 RD01
99,germano,10350,G02-03CV004 ME01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...,SAMARCO,GERMANO,G02,BENEFICIAMENTO,TRANSPORTADORES,G02-03CV004,G02-03CV004 ME01
100,germano,10641,G02-07VE002 SP01 (IMx-8 PORTÁTIL),SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,SAMARCO,GERMANO,G02,BENEFICIAMENTO,SOPRADORES,G02-07VE002 (IMx-8 PORTÁTIL),G02-07VE002 SP01 (IMx-8 PORTÁTIL)


## **3.3. DataFrame**

In [ ]:
# Selecionar colunas
df_machine = df_machines[['origem', 'id', 'name', 'path']].copy()

# Visualizar
display(df_machine.head(5))

,origem,id,name,path
0,ubu,5,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002
1,ubu,50,VENTILADOR - MANCAL LA (RANGE FIXO),SAMARCO\UBU\USINA 4\U04-06VT002\VENTILADOR - M...
2,ubu,51,VENTILADOR - MANCAL LOA (RANGE FIXO),SAMARCO\UBU\USINA 4\U04-06VT002\VENTILADOR - M...
3,ubu,116,CV16,SAMARCO\UBU\USINA 4\FILTRAGEM\CV00 - BOMBAS VÁ...
4,ubu,274,U04-06TP013,SAMARCO\UBU\USINA 4\PENEIRAMENTO\U04-06TP013
...,...,...,...,...
97,germano,10301,G02-03CV004 TA01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...
98,germano,10315,G02-03CV004 RD01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...
99,germano,10350,G02-03CV004 ME01,SAMARCO\GERMANO\G02\BENEFICIAMENTO\TRANSPORTAD...
100,germano,10641,G02-07VE002 SP01 (IMx-8 PORTÁTIL),SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...


In [ ]:
# download excel
df_machine.to_excel('cda_online_skf_machines.xlsx', index=False)

## **3.4. Lista de ID's**

In [ ]:
# Extrair lista de IDs
machine_ids = df_machine['id'].tolist()

# Exibir
print(f"{len(machine_ids)} ativos encontrados")
print(machine_ids)

85 ativos encontrados
[5, 50, 51, 116, 274, 359, 385, 402, 5305, 5309, 5424, 5456, 7175, 9407, 9418, 9429, 9440, 9560, 9561, 9798, 58, 130, 415, 1765, 1821, 1866, 2204, 2216, 2217, 2218, 2219, 2220, 2221, 2596, 2619, 2970, 3019, 3067, 3115, 3185, 3255, 3325, 3395, 3465, 3601, 3699, 3881, 3935, 3948, 5141, 5215, 5228, 5300, 5401, 5538, 6140, 6194, 6248, 6302, 6356, 7412, 7451, 7485, 7519, 7557, 7676, 7730, 7784, 7838, 7892, 7946, 8012, 9187, 9229, 9271, 9361, 9468, 9483, 9507, 9741, 10301, 10315, 10350, 10641, 10651]


## **3.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_machines"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_machine)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!


# **4. REQUISIÇÃO: SUBMACHINES**

## **4.1. Execução**

In [ ]:
def get_submachines(data, parent_name=None, submachines=None):
    if submachines is None:
        submachines = []

    for node in data:
        nome = node.get("name")

        if node.get("typeName") == "SubMachine":
            submachines.append({
                "id": node.get("id"),
                "name": nome,
                "description": node.get("description"),
                "parent": parent_name,
                "path": node.get("path"),
                "active": node.get("active"),
                "status": node.get("status")
            })

        if node.get("children"):
            get_submachines(node["children"], parent_name=nome, submachines=submachines)

    return submachines

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    data_raw = obter_arvore_hierarquia(token, base_url)
    submachines = get_submachines(data_raw)

    df = pd.DataFrame(submachines)
    df["origem"] = origem

    dfs.append(df)

df_submachines = pd.concat(dfs, ignore_index=True)

## **4.2. Estrutura e organização**

In [ ]:
# Garantir string
df_submachines["path"] = df_submachines["path"].fillna("").astype(str)

# Split
df_path = df_submachines["path"].str.split(r"\\", expand=True)
df_path.columns = [f"path_{i}" for i in range(df_path.shape[1])]

# Estrutura
df_submachines = pd.concat(
    [df_submachines[['origem', 'id', 'name', 'description', 'parent', 'path', 'active', 'status']], df_path],
    axis=1
)

# Normalizar
df_submachines["path_1"] = df_submachines["path_1"].str.strip().str.upper()

# Filtro
mapa = {"ubu": "UBU", "germano": "GERMANO"}
df_submachines = df_submachines[
    df_submachines["path_1"] == df_submachines["origem"].map(mapa)
]

# Exibir
display(df_submachines.head(1000))

,origem,id,name,description,parent,path,active,status,path_0,path_1,path_2,path_3,path_4,path_5,path_6,path_7
0,ubu,6,VENTILADOR_MANCAL LA,,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002\VE...,True,"[2, 13]",SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,VENTILADOR_MANCAL LA,None,None
1,ubu,9,VENTILADOR_MANCAL LOA,,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002\VE...,True,"[2, 13]",SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,VENTILADOR_MANCAL LOA,None,None
2,ubu,5310,U04-06FN001-ME PRINCIPAL,,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,True,[28],SAMARCO,UBU,USINA 4,ENDURECIMENTO,U04-06FN001,U04-06FN001-ME PRINCIPAL,None,None
3,ubu,7201,U04-06FN001-BB Lubrif,MOTOR - BOMBA DE LUBRIFICAÇÃO,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,True,[1],SAMARCO,UBU,USINA 4,ENDURECIMENTO,U04-06FN001,U04-06FN001-BB Lubrif,None,None
4,ubu,5311,U04-06FN001-RD1,,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,True,"[8, 9, 12]",SAMARCO,UBU,USINA 4,ENDURECIMENTO,U04-06FN001,U04-06FN001-RD1,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,germano,7520,M03-02CP004 ME,,M03-02CP004 (GW12),SAMARCO\GERMANO\MINERODUTO\EB007\COMPRESSORES\...,True,[28],SAMARCO,GERMANO,MINERODUTO,EB007,COMPRESSORES,M03-02CP004 (GW12),M03-02CP004 ME,None
269,germano,7523,M03-02CP004 COMPRESSOR,,M03-02CP004 (GW12),SAMARCO\GERMANO\MINERODUTO\EB007\COMPRESSORES\...,True,"[8, 28]",SAMARCO,GERMANO,MINERODUTO,EB007,COMPRESSORES,M03-02CP004 (GW12),M03-02CP004 COMPRESSOR,None
270,germano,7558,MOTOR,,M03-02BP002 (GW28),SAMARCO\GERMANO\MINERODUTO\EB007\M03-02BP002 (...,True,"[2, 28]",SAMARCO,GERMANO,MINERODUTO,EB007,M03-02BP002 (GW28),MOTOR,None,None
271,germano,7569,REDUTOR,,M03-02BP002 (GW28),SAMARCO\GERMANO\MINERODUTO\EB007\M03-02BP002 (...,True,[2],SAMARCO,GERMANO,MINERODUTO,EB007,M03-02BP002 (GW28),REDUTOR,None,None


## **4.3. DataFrame**

In [ ]:
df_submachines = df_submachines[['origem', 'id', 'name', 'description', 'parent', 'path', 'status', 'active']]

display(df_submachines.head(5))

,origem,id,name,description,parent,path,status,active
0,ubu,6,VENTILADOR_MANCAL LA,,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002\VE...,"[2, 13]",True
1,ubu,9,VENTILADOR_MANCAL LOA,,U04-06VT002,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002\VE...,"[2, 13]",True
2,ubu,5310,U04-06FN001-ME PRINCIPAL,,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,[28],True
3,ubu,7201,U04-06FN001-BB Lubrif,MOTOR - BOMBA DE LUBRIFICAÇÃO,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,[1],True
4,ubu,5311,U04-06FN001-RD1,,U04-06FN001,SAMARCO\UBU\USINA 4\ENDURECIMENTO\U04-06FN001\...,"[8, 9, 12]",True
...,...,...,...,...,...,...,...,...
268,germano,7520,M03-02CP004 ME,,M03-02CP004 (GW12),SAMARCO\GERMANO\MINERODUTO\EB007\COMPRESSORES\...,[28],True
269,germano,7523,M03-02CP004 COMPRESSOR,,M03-02CP004 (GW12),SAMARCO\GERMANO\MINERODUTO\EB007\COMPRESSORES\...,"[8, 28]",True
270,germano,7558,MOTOR,,M03-02BP002 (GW28),SAMARCO\GERMANO\MINERODUTO\EB007\M03-02BP002 (...,"[2, 28]",True
271,germano,7569,REDUTOR,,M03-02BP002 (GW28),SAMARCO\GERMANO\MINERODUTO\EB007\M03-02BP002 (...,[2],True


In [ ]:
# download excel
df_submachines.to_excel('cda_online_skf_submachines.xlsx', index=False)

## **4.4. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_submachines"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_submachines)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!


# **5. REQUISIÇÃO: MACHINE PART**

## **5.1. Execução**

In [ ]:
def get_machine_parts(token, base_url, machine_ids, origem):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    registros = []

    for mid in machine_ids:
        url = f"{base_url}/v1/machines/{mid}/parts"
        resp = requests.get(url, headers=headers, verify=False)

        if resp.status_code == 200:
            data = resp.json()

            for part in data:
                fault_list = part.get("faultFrequencies", [])

                if not fault_list:
                    registros.append({
                        "MachineId": mid,
                        "PartID": part.get("id"),
                        "PartName": part.get("name"),
                        "Type": part.get("type"),
                        "Ratio": part.get("ratio"),
                        "Brand": part.get("brand"),
                        "Typeno": part.get("typeno"),
                        "SpeedPointId": part.get("speedPointId"),
                        "Name": None,
                        "Multiple": None,
                        "origem": origem
                    })
                else:
                    for f in fault_list:
                        registros.append({
                            "MachineId": mid,
                            "PartID": part.get("id"),
                            "PartName": part.get("name"),
                            "Type": part.get("type"),
                            "Ratio": part.get("ratio"),
                            "Brand": part.get("brand"),
                            "Typeno": part.get("typeno"),
                            "SpeedPointId": part.get("speedPointId"),
                            "Name": f.get("name"),
                            "Multiple": f.get("multiple"),
                            "origem": origem
                        })

    return pd.DataFrame(registros)

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    machine_ids_origem = df_machines[
        df_machines["origem"] == origem
    ]["id"].tolist()

    df = get_machine_parts(token, base_url, machine_ids_origem, origem)
    dfs.append(df)

df_parts = pd.concat(dfs, ignore_index=True)

df_parts.head(10000)

,MachineId,PartID,PartName,Type,Ratio,Brand,Typeno,SpeedPointId,Name,Multiple,origem
0,5,1,Mancal de deslizamento Ventilador LOA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu
1,5,2,Eixo Ventilador,Shaft,1.0,None,None,7,Eixo Ventilador,1.00,ubu
2,5,3,Mancal de deslizamento Ventilador LA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu
3,5,4,Acoplamento,Fixed coupling,1.0,None,None,7,Acoplamento,1.00,ubu
4,5,5,Mancal de deslizamento Motor LA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu
...,...,...,...,...,...,...,...,...,...,...,...
3016,10651,2092,Eixo Motor,Shaft,1.0,None,None,10650,Eixo Motor,1.00,germano
3017,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BPFO,3.10,germano
3018,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BPFI,4.90,germano
3019,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BSF,2.10,germano


In [ ]:
# download excel
df_parts.to_excel('cda_online_skf_parts.xlsx', index=False)

## **5.2. Estrutura e organização**

In [ ]:
# merge
df_parts = df_parts.merge(
    df_machines[["id", "path"]],
    left_on="MachineId",
    right_on="id",
    how="left"
)

# remover coluna duplicada do merge
df_parts = df_parts.drop(columns=["id"], errors="ignore")

# garantir string
df_parts["path"] = df_parts["path"].fillna("").astype(str)

# split
df_path = df_parts["path"].str.split(r"\\", expand=True)
df_path.columns = [f"path_{i}" for i in range(df_path.shape[1])]

df_parts = pd.concat([df_parts, df_path], axis=1)

# garantir path_1 válido
df_parts["path_1"] = df_parts.get("path_1", "").astype(str).str.strip().str.upper()

# filtro
mapa = {"ubu": "UBU", "germano": "GERMANO"}

df_parts = df_parts[
    df_parts["path_1"] == df_parts["origem"].map(mapa)
]

# exibir
df_parts.head(10)

,MachineId,PartID,PartName,Type,Ratio,Brand,Typeno,SpeedPointId,Name,Multiple,origem,path,path_0,path_1,path_2,path_3,path_4,path_5,path_6
0,5,1,Mancal de deslizamento Ventilador LOA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
1,5,2,Eixo Ventilador,Shaft,1.0,None,None,7,Eixo Ventilador,1.0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
2,5,3,Mancal de deslizamento Ventilador LA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
3,5,4,Acoplamento,Fixed coupling,1.0,None,None,7,Acoplamento,1.0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
4,5,5,Mancal de deslizamento Motor LA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
5,5,6,Eixo Motor,Shaft,1.0,None,None,7,Eixo Motor,1.0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
6,5,7,Mancal de deslizamento Motor LOA,Sleeve bearing,1.0,None,None,7,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
7,5,8,Tacho,Speed input location,1.0,None,None,7,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
8,5,36,Rotor do Ventilador,Impeller,1.0,None,None,7,Rotor do Ventilador,13.0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002,None,None
9,50,37,Rotação 1,Speed input location,1.0,None,None,0,None,NaN,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\VENTILADOR - M...,SAMARCO,UBU,USINA 4,U04-06VT002,VENTILADOR - MANCAL LA (RANGE FIXO),None,None


## **5.3. DataFrame**

In [ ]:
colunas = [
    "origem", "path", "MachineId", "PartID", "PartName", "Type", "Ratio",
    "Brand", "Typeno", "SpeedPointId",
    "Name", "Multiple"
]

df_parts = df_parts[colunas].drop_duplicates()

display(df_parts.head(5))

,origem,path,MachineId,PartID,PartName,Type,Ratio,Brand,Typeno,SpeedPointId,Name,Multiple
0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,1,Mancal de deslizamento Ventilador LOA,Sleeve bearing,1.0,None,None,7,None,NaN
1,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,2,Eixo Ventilador,Shaft,1.0,None,None,7,Eixo Ventilador,1.00
2,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,3,Mancal de deslizamento Ventilador LA,Sleeve bearing,1.0,None,None,7,None,NaN
3,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,4,Acoplamento,Fixed coupling,1.0,None,None,7,Acoplamento,1.00
4,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,5,Mancal de deslizamento Motor LA,Sleeve bearing,1.0,None,None,7,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3016,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,2092,Eixo Motor,Shaft,1.0,None,None,10650,Eixo Motor,1.00
3017,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BPFO,3.10
3018,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BPFI,4.90
3019,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,2093,6319 LA,Bearing,1.0,SKF,6319,10650,6319 LA BSF,2.10


In [ ]:
# download excel
df_parts.to_excel('cda_online_skf_machineparts.xlsx', index=False)

## **5.4. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_machineparts"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_parts)

print("Dados enviados com sucesso para o Google Sheets!")

# **6. REQUISIÇÃO: POINTS**

## **6.1. Execução**

In [ ]:
def get_points(token, base_url, machine_ids, origem):
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    registros = []

    mapa_path = df_machines.set_index("id")["path"].to_dict()

    for mid in machine_ids:
        url = f"{base_url}/v1/machines/{mid}/points"
        resp = requests.get(url, headers=headers, verify=False)

        if resp.status_code != 200:
            continue

        registros.append({
            "path": mapa_path.get(mid),
            "MachineId": mid,
            "origem": origem,
            "data": resp.json()
        })

    return pd.DataFrame(registros)

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    machine_ids_origem = df_machines[
        df_machines["origem"] == origem
    ]["id"].tolist()

    df_temp = get_points(token, base_url, machine_ids_origem, origem)
    dfs.append(df_temp)

df_points = pd.concat(dfs, ignore_index=True)

## **6.2. Estrutura e organização**

In [ ]:
def explode_points(df_raw):
    registros = []

    for _, row in df_raw.iterrows():
        for p in row["data"]:
            registros.append({
                "origem": row["origem"],
                "path": row["path"],
                "MachineID": row["MachineId"],
                "SubmachineID": p.get("ParentID"),
                "ID": p.get("ID"),
                "Name": p.get("Name"),
                "Description": p.get("Description"),
                "NodeTypeName": p.get("NodeTypeName"),
                "EU": p.get("EU"),
                "DetectionName": p.get("DetectionName")
            })

    return pd.DataFrame(registros)

df_points = explode_points(df_points)

## **6.3. DataFrame**

In [ ]:
display(df_points.head(5))

,origem,path,MachineID,SubmachineID,ID,Name,Description,NodeTypeName,EU,DetectionName
0,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,6,11,OS 03XYSC DE,Linha de Centro Ventilador LA,Shaft centerline (IMx),um,
1,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,9,12,OS 04XYSC NDE,Linha de Centro Mancal LOA,Shaft centerline (IMx),um,
2,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,6,31,OS 03XD DE 100 Hz,,Dynamic (IMx),um,PtP
3,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,6,17,OS 03XD DE,Deslocamento X mancal LA,Dynamic (IMx),um,PtP
4,ubu,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,5,6,8,OS 03XYOR DE,Órbita Ventilador LA,Dynamic (IMx),um,PtP
...,...,...,...,...,...,...,...,...,...,...
3043,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,10651,10657,OS 01HV,Motor LOA - Velocidade Horizontal,Dynamic (IMx),mm/s,rms
3044,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,10651,10658,OS 01HE2,Motor LOA - Envelope 2 Horizontal,"Dynamic, Envelope (IMx)",gE,PtP
3045,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,10651,10656,OS 01HE3,Motor LOA - Envelope 3 Horizontal,"Dynamic, Envelope (IMx)",gE,PtP
3046,germano,SAMARCO\GERMANO\G02\BENEFICIAMENTO\SOPRADORES\...,10651,10651,10652,OS 01HA,Motor LOA - Aceleração Horizontal,Dynamic (IMx),g,Peak


In [ ]:
# download excel
df_points.to_excel('cda_online_skf_points.xlsx', index=False)

## **6.4. Lista**

In [ ]:
# Extrai lista com os IDs dos ativos filtrados
point_ids = df_points['ID'].tolist()

# Exibe a lista
print(f"{len(point_ids)} pontos encontrados")
print(point_ids)

3048 pontos encontrados
[11, 12, 31, 17, 8, 26, 20, 25, 36, 32, 27, 28, 48, 43, 44, 45, 47, 34, 29, 37, 10, 41, 35, 33, 30, 42, 7, 7174, 53, 52, 54, 55, 5108, 129, 128, 127, 126, 124, 123, 122, 121, 346, 347, 345, 344, 341, 342, 340, 339, 335, 336, 334, 333, 330, 331, 329, 328, 298, 299, 297, 296, 303, 304, 302, 301, 293, 294, 292, 291, 283, 284, 282, 281, 378, 379, 377, 376, 383, 384, 382, 381, 373, 374, 372, 371, 368, 369, 367, 366, 395, 396, 394, 393, 400, 401, 399, 398, 412, 413, 411, 410, 407, 408, 406, 405, 9482, 9481, 7674, 9465, 9151, 9150, 7673, 7172, 7173, 5308, 5307, 5306, 5399, 5320, 5319, 6127, 6126, 6123, 6122, 5398, 5397, 5395, 5329, 5392, 5390, 5389, 5388, 5385, 5384, 5381, 5378, 5375, 5373, 5371, 5363, 5349, 5358, 5337, 5334, 7204, 7203, 5365, 5380, 5383, 7266, 5351, 5348, 5354, 7273, 7272, 5318, 5366, 5357, 5353, 6128, 7271, 7270, 6124, 7269, 7268, 7267, 5396, 5393, 5387, 5377, 5372, 5370, 5369, 5368, 5367, 5364, 5356, 5352, 5350, 5346, 5335, 5344, 5341, 5340, 5339, 5

## **6.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_points"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_points)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!


# **7. REQUISIÇÃO: ALARMS**

## **7.1. Execução**

In [ ]:
def get_alarms(token, base_url, machine_ids, origem):

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    registros = []

    for mid in machine_ids:
        url = f"{base_url}/v1/machines/{mid}/points"
        resp = requests.get(url, headers=headers, verify=False)

        if resp.status_code != 200:
            continue

        for ponto in resp.json():

            if not isinstance(ponto, dict):
                continue

            registro = {
                "origem": origem,
                "MachineId": mid,
                "ID": ponto.get("ID"),
                "HighAlarm": None,
                "HighWarning": None,
                "Freq_AlarmLevel": None,
                "Freq_WarningLevel": None
            }

            overall_alarm = ponto.get("OverallAlarm") or {}
            summary = overall_alarm.get("Summary", "")

            if summary:
                parts = summary.lower().replace(" / ", "/").split("/")

                for part in parts:
                    try:
                        val = float(part.split()[-1].replace(",", "."))
                    except:
                        val = None

                    if "high alarm" in part:
                        registro["HighAlarm"] = val
                    elif "high warning" in part:
                        registro["HighWarning"] = val

            freq_list = ponto.get("Frequencies", [])
            freq_overall = next(
                (f for f in freq_list if f.get("Frequency") == "Overall"),
                {}
            )

            try:
                registro["Freq_AlarmLevel"] = float(
                    str(freq_overall.get("AlarmLevel", "")).split()[0].replace(",", ".")
                )
            except:
                pass

            try:
                registro["Freq_WarningLevel"] = float(
                    str(freq_overall.get("WarningLevel", "")).split()[0].replace(",", ".")
                )
            except:
                pass

            registros.append(registro)

    df = pd.DataFrame(registros)

    cols = ["HighAlarm", "HighWarning", "Freq_AlarmLevel", "Freq_WarningLevel"]
    df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

    df["ID"] = pd.to_numeric(df["ID"], errors="coerce").astype("Int64")

    return df

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    machine_ids_origem = df_machines[
        df_machines["origem"] == origem
    ]["id"].tolist()

    df_temp = get_alarms(token, base_url, machine_ids_origem, origem)
    dfs.append(df_temp)

df_alarms = pd.concat(dfs, ignore_index=True)

## **7.2. DataFrame**

In [ ]:
print(f"Total de registros: {len(df_alarms)}")

display(df_alarms.head(5))

Total de registros: 3048


,origem,MachineId,ID,HighAlarm,HighWarning,Freq_AlarmLevel,Freq_WarningLevel
0,ubu,5,11,NaN,NaN,NaN,NaN
1,ubu,5,12,NaN,NaN,NaN,NaN
2,ubu,5,31,NaN,NaN,90.0,70.0
3,ubu,5,17,NaN,NaN,90.0,70.0
4,ubu,5,8,NaN,NaN,100.0,85.0
...,...,...,...,...,...,...,...
3043,germano,10651,10657,NaN,NaN,7.0,5.0
3044,germano,10651,10658,NaN,NaN,7.0,2.8
3045,germano,10651,10656,NaN,NaN,10.0,4.0
3046,germano,10651,10652,NaN,NaN,3.0,2.5


In [ ]:
# download excel
df_alarms.to_excel('cda_online_skf_alarms.xlsx', index=False)

## **7.3. Carga de Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_alarms"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_alarms)

print("Dados enviados com sucesso para o Google Sheets!")

Dados enviados com sucesso para o Google Sheets!


# **8. REQUISIÇÃO: LAST MEASUREMENTS**

## **8.1. Execução**

In [ ]:
def get_measurements(token, base_url, machine_ids, origem):

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    registros = []

    for mid in machine_ids:
        url = f"{base_url}/v1/machines/{mid}/points?IncludeLastMeasurement=true"
        resp = requests.get(url, headers=headers, verify=False)

        if resp.status_code != 200:
            continue

        for ponto in resp.json():

            last = ponto.get("LastMeasurement") or {}

            measurements = last.get("Measurements") or [None]

            for m in measurements:
                registros.append({
                    "ReadingTimeUTC": last.get("ReadingTimeUTC"),
                    "PointID": ponto.get("ID"),
                    "Speed": last.get("Speed"),
                    "SpeedUnits": last.get("SpeedUnits"),
                    "Direction": m.get("Direction") if m else None,
                    "ChannelName": m.get("ChannelName") if m else None,
                    "Level": m.get("Level") if m else None,
                    "Units": m.get("Units") if m else None,
                    "origem": origem
                })

    return pd.DataFrame(registros)

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    machine_ids_origem = df_machines[
        df_machines["origem"] == origem
    ]["id"].tolist()

    df_temp = get_measurements(token, base_url, machine_ids_origem, origem)
    dfs.append(df_temp)

df_lastmeasurements = pd.concat(dfs, ignore_index=True)

## **8.2. DataFrame**

In [ ]:
df_lastmeasurements["ReadingTimeUTC"] = pd.to_datetime(
    df_lastmeasurements["ReadingTimeUTC"],
    errors="coerce"
)

df_lastmeasurements = df_lastmeasurements[
    df_lastmeasurements["ReadingTimeUTC"].notna()
]

df_lastmeasurements.head(5)

,ReadingTimeUTC,PointID,Speed,SpeedUnits,Direction,ChannelName,Level,Units,origem
0,2025-05-29 06:46:16.710,11,0.000000,RPM,X,Overall,870.765259,um,ubu
1,2025-06-18 16:12:42.150,12,0.000000,RPM,X,Overall,738.065796,um,ubu
2,2025-12-07 01:00:13.910,31,324.398956,RPM,X,1 X RPM,2.881257,um PtP,ubu
3,2025-12-07 01:00:13.910,31,324.398956,RPM,X,Valor global,76.171982,um PtP,ubu
4,2025-12-07 01:00:30.120,17,324.624146,RPM,X,1 X RPM,30.363472,um PtP,ubu
...,...,...,...,...,...,...,...,...,...
3071,2026-04-22 14:30:31.190,10653,0.000000,RPM,X,Valor global,0.013327,g P,germano
3072,2026-04-22 14:30:08.540,10657,0.000000,RPM,X,Valor global,0.941200,mm/s rms,germano
3073,2026-04-22 14:30:24.820,10658,0.000000,RPM,X,Valor global,0.003552,gE PtP,germano
3074,2026-04-22 14:30:05.290,10656,0.000000,RPM,X,Valor global,0.004831,gE PtP,germano


In [ ]:
# download excel
df_lastmeasurements.to_excel('cda_online_skf_lastmeasurements.xlsx', index=False)

## **8.3. Carga de Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_lastmeasurements"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_lastmeasurements)

print("Dados enviados com sucesso para o Google Sheets!")

# **9. REQUISIÇÃO: MEASUREMENTS**

## **9.1. Execução**

In [ ]:
def consultar_trends(point_ids, token, base_url, origem):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    now_utc = datetime.now(timezone.utc)

    ontem_inicio = (now_utc - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
    ontem_fim = (now_utc - timedelta(days=1)).replace(hour=23, minute=59, second=59, microsecond=0)

    params = {
        "fromDateUTC": ontem_inicio.isoformat().replace("+00:00", "Z"),
        "toDateUTC": ontem_fim.isoformat().replace("+00:00", "Z")
    }

    resultados = []

    for pid in point_ids:
        url = f"{base_url}/v1/points/{int(pid)}/trendMeasurements"
        response = requests.get(url, headers=headers, params=params, verify=False)

        if response.status_code == 200:
            data = response.json()

            for record in data:
                base = {
                    "ReadingTimeUTC": record.get("ReadingTimeUTC"),
                    "PointID": record.get("PointID"),
                    "origem": origem
                }

                for m in record.get("Measurements", []):
                    resultados.append({
                        **base,
                        "ChannelName": m.get("ChannelName"),
                        "Direction": m.get("Direction"),
                        "Level": m.get("Level"),
                        "Units": m.get("Units")
                    })

    return pd.DataFrame(resultados)

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    point_ids_origem = df_points[
        df_points["origem"] == origem
    ]["ID"].dropna().astype(int).tolist()

    df = consultar_trends(point_ids_origem, token, base_url, origem)
    dfs.append(df)

df_trends = pd.concat(dfs, ignore_index=True)

## **9.2. DataFrame**

In [ ]:
df_trends = df_trends[
    (df_trends['ChannelName'].isin(['Valor global', 'Overall'])) &
    (df_trends['Direction'] == "X")
]

colunas = ['ReadingTimeUTC', 'PointID', 'Level', 'origem']

df_trendMeasurements = df_trends[colunas].drop_duplicates()

print(f"{len(df_trendMeasurements)} medições finais")
display(df_trendMeasurements.head(5))

36911 medições finais


,ReadingTimeUTC,PointID,Level,origem
0,2026-04-22T23:09:50.05,5399,0.045723,ubu
1,2026-04-22T22:04:59.63,5399,0.041980,ubu
2,2026-04-22T21:00:09.19,5399,0.041092,ubu
3,2026-04-22T20:11:31.31,5399,0.042610,ubu
4,2026-04-22T19:06:40.67,5399,0.044868,ubu
...,...,...,...,...
10821,2026-04-22T07:00:11.61,9744,0.021686,ubu
10822,2026-04-22T06:00:02.03,9744,0.000450,ubu
10823,2026-04-22T05:00:09.27,9744,0.017233,ubu
10824,2026-04-22T04:00:16.55,9744,0.019092,ubu


In [ ]:
# download excel
df_trendMeasurements.to_excel('cda_online_skf_measurements.xlsx', index=False)

## **9.3. Carga de Sheets**

In [ ]:
# Nome da planilha e aba
nome_da_planilha = "cda_online_skf_measurements"
nome_da_aba = "Sheet1"

# Abre a planilha e aba
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Lê os dados atuais da aba
df_existente = get_as_dataframe(aba, evaluate_formulas=True).dropna(how="all")

# Colunas obrigatórias
colunas_chave = ['ReadingTimeUTC', 'PointID', 'Level']

# Se a aba está vazia ou não tem as colunas necessárias → cria cabeçalho
if df_existente.empty or not all(col in df_existente.columns for col in colunas_chave):
    aba.clear()
    set_with_dataframe(aba, pd.DataFrame(columns=colunas_chave), row=1, col=1, include_column_header=True)
    df_existente = pd.DataFrame(columns=colunas_chave)

# Garante que colunas estão no mesmo formato e ordem
df_existente = df_existente[colunas_chave].dropna()

# Remove duplicados e encontra apenas as linhas novas
df_novos = df_trendMeasurements[~df_trendMeasurements.isin(df_existente.to_dict(orient='list')).all(axis=1)]

# Se houver novos registros, adiciona abaixo
if not df_novos.empty:
    # Número de linhas já existentes (para inserir a partir da próxima linha vazia)
    ultima_linha = len(df_existente) + 2  # +1 para header, +1 para próxima
    set_with_dataframe(aba, df_novos, row=ultima_linha, col=1, include_column_header=False)
    print(f"{len(df_novos)} novas medições adicionadas à planilha!")
else:
    print("Nenhuma medição nova para inserir — tudo já está na planilha.")

5831 novas medições adicionadas à planilha!


## **9.4. Tratamento de duplicatas**

In [ ]:
# Nome da planilha e aba
nome_da_planilha = "cda_online_skf_measurements"
nome_da_aba = "Sheet1"

# Abre a planilha e aba
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Lê os dados da aba
df = get_as_dataframe(aba, evaluate_formulas=True).dropna(how="all")

# Remove duplicatas com base nas colunas chave
colunas_chave = ['ReadingTimeUTC', 'PointID', 'Level', 'Units']
df_limpo = df.drop_duplicates(subset=colunas_chave, keep='first')

# Limpa aba (opcional, mas garante que não fica lixo antigo abaixo)
aba.clear()

# Reescreve os dados limpos na planilha (com cabeçalho)
set_with_dataframe(aba, df_limpo, include_column_header=True)

print(f"Removidas {len(df) - len(df_limpo)} duplicatas. Planilha atualizada com {len(df_limpo)} registros únicos.")

Removidas 8162 duplicatas. Planilha atualizada com 95272 registros únicos.


# **10. REQUISIÇÃO: NOTES**

## **10.1. Execução**

In [ ]:
def get_notes(token, base_url, origem):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    url = f"{base_url}/v1/notes"
    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    data = response.json()

    registros = []
    for note in data:
        registros.append({
            "NoteID": note.get("idNote"),
            "PointID": note.get("idNode"),
            "NodeName": note.get("nodeName"),
            "CreatedAt": note.get("noteDateUTC"),
            "Title": note.get("title"),
            "Author": note.get("signature"),
            "Text": note.get("noteComment"),
            "Priority": note.get("priority"),
            "origem": origem
        })

    return pd.DataFrame(registros)

dfs = []

for origem, base_url in BASE_URLS.items():
    token = obter_token(base_url)

    df = get_notes(token, base_url, origem)
    dfs.append(df)

df_notes = pd.concat(dfs, ignore_index=True)

## **10.2. Estrutura e organização**

In [ ]:
df_notes = df_notes.merge(
    df_points[["ID", "path"]],
    left_on="PointID",
    right_on="ID",
    how="left"
)

# Garantir string
df_notes["path"] = df_notes["path"].fillna("").astype(str)

# Split
df_path = df_notes["path"].str.split(r"\\", expand=True)
df_path.columns = [f"path_{i}" for i in range(df_path.shape[1])]

df_notes = pd.concat([df_notes, df_path], axis=1)

# Normalizar
df_notes["path_1"] = df_notes["path_1"].str.strip().str.upper()

# Filtro
mapa = {"ubu": "UBU", "germano": "GERMANO"}

df_notes = df_notes[
    df_notes["path_1"] == df_notes["origem"].map(mapa)
]

df_notes.head(10)

,NoteID,PointID,NodeName,CreatedAt,Title,Author,Text,Priority,origem,ID,path,path_0,path_1,path_2,path_3,path_4
2,25,5308,OS 04XYSC NDE (26/04/25)(Copiar),2025-04-26T17:30:40,Manutenção e Ajuste de Sensor,Adilson Hilário Gonçalves,Realizado ajuste e calibração dos sensores pro...,0,ubu,5308.0,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,SAMARCO,UBU,USINA 4,U04-06VT002,CENTER LINE - GAPVolt
3,24,5307,OS 04XYSC NDE (26/04/25),2025-04-26T17:30:10,Manutenção e Ajuste de Sensor,Adilson Hilário Gonçalves,Realizado ajuste e calibração dos sensores pro...,0,ubu,5307.0,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,SAMARCO,UBU,USINA 4,U04-06VT002,CENTER LINE - GAPVolt
4,23,5306,OS 03XYSC DE (26/04/25),2025-04-26T17:30:00,Manutenção e Ajuste de Sensor,Adilson Hilário Gonçalves,Foi realizado inspeção dos casquilhos do manca...,0,ubu,5306.0,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,SAMARCO,UBU,USINA 4,U04-06VT002,CENTER LINE - GAPVolt
12,12,35,OS 04Y GAP um,2023-11-27T16:10:51,Registro de Alarme,Romenig Silva dos Reis,Ponto 04Y GAP variando a tendência e vibração ...,0,ubu,35.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
13,11,36,OS 03X GAP Volts,2023-11-14T11:23:49,Registro de Alarme,Romenig Silva dos Reis,VT002 foi parado devido a desligamento da rede...,0,ubu,36.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
14,10,32,OS 03X GAP um,2023-11-08T16:14:20,Registro de Alarme,Romenig Silva dos Reis,Tendência apresentando queda repentina possíve...,0,ubu,32.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
17,7,33,OS 03Y GAP um,2023-06-26T19:11:00,Registro de Alarme,Romenig Silva dos Reis,"26/06/23 as 15hs, houve redução da rotação dev...",0,ubu,33.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
18,6,35,OS 04Y GAP um,2023-06-22T11:25:17,Registro de Alarme,Romenig Silva dos Reis,Houve parada da usina no dia 21/06 devido a pr...,0,ubu,35.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
19,5,32,OS 03X GAP um,2023-06-22T11:23:53,Registro de Alarme,Romenig Silva dos Reis,Houve parada da usina no dia 21/06 devido a pr...,0,ubu,32.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002
20,4,35,OS 04Y GAP um,2023-05-20T19:59:26,Registro de Alarme,Bruno Arthur Celenza Bellini,Em 03/04/2023 foi realizado aferição da distan...,0,ubu,35.0,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,SAMARCO,UBU,USINA 4,U04-06VT002,U04-06VT002


## **10.3. DataFrame**

In [ ]:
colunas = [
    "NoteID", "path", "PointID", "NodeName", "Author",
    "CreatedAt", "Title", "Text",
    "Priority", "origem"
]

df_notes = df_notes[colunas].drop_duplicates()

display(df_notes.head(5))

,NoteID,path,PointID,NodeName,Author,CreatedAt,Title,Text,Priority,origem
2,25,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,5308,OS 04XYSC NDE (26/04/25)(Copiar),Adilson Hilário Gonçalves,2025-04-26T17:30:40,Manutenção e Ajuste de Sensor,Realizado ajuste e calibração dos sensores pro...,0,ubu
3,24,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,5307,OS 04XYSC NDE (26/04/25),Adilson Hilário Gonçalves,2025-04-26T17:30:10,Manutenção e Ajuste de Sensor,Realizado ajuste e calibração dos sensores pro...,0,ubu
4,23,SAMARCO\UBU\USINA 4\U04-06VT002\CENTER LINE - ...,5306,OS 03XYSC DE (26/04/25),Adilson Hilário Gonçalves,2025-04-26T17:30:00,Manutenção e Ajuste de Sensor,Foi realizado inspeção dos casquilhos do manca...,0,ubu
12,12,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,35,OS 04Y GAP um,Romenig Silva dos Reis,2023-11-27T16:10:51,Registro de Alarme,Ponto 04Y GAP variando a tendência e vibração ...,0,ubu
13,11,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,36,OS 03X GAP Volts,Romenig Silva dos Reis,2023-11-14T11:23:49,Registro de Alarme,VT002 foi parado devido a desligamento da rede...,0,ubu
14,10,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,32,OS 03X GAP um,Romenig Silva dos Reis,2023-11-08T16:14:20,Registro de Alarme,Tendência apresentando queda repentina possíve...,0,ubu
17,7,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,33,OS 03Y GAP um,Romenig Silva dos Reis,2023-06-26T19:11:00,Registro de Alarme,"26/06/23 as 15hs, houve redução da rotação dev...",0,ubu
18,6,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,35,OS 04Y GAP um,Romenig Silva dos Reis,2023-06-22T11:25:17,Registro de Alarme,Houve parada da usina no dia 21/06 devido a pr...,0,ubu
19,5,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,32,OS 03X GAP um,Romenig Silva dos Reis,2023-06-22T11:23:53,Registro de Alarme,Houve parada da usina no dia 21/06 devido a pr...,0,ubu
20,4,SAMARCO\UBU\USINA 4\U04-06VT002\U04-06VT002,35,OS 04Y GAP um,Bruno Arthur Celenza Bellini,2023-05-20T19:59:26,Registro de Alarme,Em 03/04/2023 foi realizado aferição da distan...,0,ubu


In [ ]:
# download excel
df_notes.to_excel('cda_online_skf_notes.xlsx', index=False)

## **10.4. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_online_skf_notes"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_notes)

print("Dados enviados com sucesso para o Google Sheets!")

# **11. PSEUDOCÓDIGO**

# CDA Online SKF

---

## 1. CONFIGURAÇÕES INICIAIS

```
DEFINIR CREDENTIALS = { username, password, grant_type }

DEFINIR BASE_URLS = {
    "ubu"     : "http://services.repcenter.skf.com:22011",
    "germano" : "http://services.repcenter.skf.com:20446"
}

FUNÇÃO build_urls(base_url):
    RETORNAR dicionário com endpoints:
        token, machines, parts, submachines, points, notes

AUTENTICAR no Google Sheets via OAuth (gspread + google.auth)
```

---

## 2. FUNÇÕES UTILITÁRIAS

### 2.1. obter_token(base_url)
```
FUNÇÃO obter_token(base_url):
    POST para base_url/token com CREDENTIALS
    SE erro: lançar exceção
    RETORNAR access_token da resposta
```

### 2.2. obter_arvore_hierarquia(token, base_url)
```
FUNÇÃO obter_arvore_hierarquia(token, base_url):
    GET para base_url/v1/hierarchy com Bearer token
    SE erro: lançar exceção
    RETORNAR JSON com a árvore hierárquica completa
```

---

## 3. MACHINES

### Busca e construção
```
FUNÇÃO get_machines(token, base_url, origem):
    GET para base_url/v1/machines com Bearer token
    df = DataFrame(resposta.json())
    adicionar coluna "origem"
    RETORNAR df

FUNÇÃO get_all_machines():
    PARA CADA (origem, base_url) EM BASE_URLS:
        token = obter_token(base_url)
        executar get_machines(token, base_url, origem)
    RETORNAR concatenação de todos os DataFrames
```

### Estrutura e organização
```
df_machines = get_all_machines()

converter coluna "path" para string
DIVIDIR "path" por "\" → colunas path_0, path_1, path_2, ...
normalizar path_1: remover espaços, converter para MAIÚSCULO

FILTRAR df_machines:
    manter apenas linhas onde path_1 == mapa[origem]
    (ex: "ubu" → "UBU", "germano" → "GERMANO")

df_machine = selecionar colunas [origem, id, name, path]
machine_ids = lista de IDs das machines filtradas

SALVAR Excel "cda_online_skf_machines.xlsx"
ENVIAR df_machine para Google Sheets "cda_online_skf_machines" (limpar + escrever)
```

---

## 4. SUBMACHINES

### Busca recursiva na hierarquia
```
FUNÇÃO get_submachines(data, parent_name, submachines=[]):
    PARA CADA nó EM data:
        SE nó.typeName == "SubMachine":
            ADICIONAR ao submachines:
                { id, name, description, parent, path, active, status }

        SE nó possui filhos (children):
            chamar get_submachines(nó.children, parent_name=nó.name, submachines)

    RETORNAR submachines
```

### Execução por origem
```
PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    data_raw = obter_arvore_hierarquia(token, base_url)
    submachines = get_submachines(data_raw)
    df = DataFrame(submachines) + coluna "origem"
    acumular em dfs

df_submachines = concatenar todos os dfs

converter "path" para string
DIVIDIR "path" por "\" → colunas path_0, path_1, ...
normalizar path_1
FILTRAR por origem (mesmo critério de machines)

selecionar colunas [origem, id, name, description, parent, path, status, active]

SALVAR Excel "cda_online_skf_submachines.xlsx"
ENVIAR para Google Sheets "cda_online_skf_submachines"
```

---

## 5. MACHINE PARTS

### Busca por machine ID
```
FUNÇÃO get_machine_parts(token, base_url, machine_ids, origem):
    PARA CADA machine_id EM machine_ids:
        GET para base_url/v1/machines/{machine_id}/parts

        SE status == 200:
            PARA CADA part na resposta:
                SE part NÃO tem faultFrequencies:
                    adicionar registro com campos da peça + Name=None, Multiple=None
                SENÃO:
                    PARA CADA frequência de falha:
                        adicionar registro com campos da peça + Name e Multiple da frequência

    RETORNAR DataFrame de registros
```

### Execução por origem
```
PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    machine_ids_origem = IDs de df_machines filtrados por origem
    df = get_machine_parts(token, base_url, machine_ids_origem, origem)
    acumular em dfs

df_parts = concatenar todos os dfs

JUNTAR df_parts com df_machines pelo MachineId → trazer coluna "path"
DIVIDIR "path" por "\" → colunas path_0, path_1, ...
normalizar e FILTRAR por origem

selecionar colunas [origem, path, MachineId, PartID, PartName, Type,
                    Ratio, Brand, Typeno, SpeedPointId, Name, Multiple]
remover duplicatas

SALVAR Excel "cda_online_skf_machineparts.xlsx"
ENVIAR para Google Sheets "cda_online_skf_machineparts"
```

---

## 6. POINTS

### Busca por machine ID
```
FUNÇÃO get_points(token, base_url, machine_ids, origem):
    mapa_path = dicionário { machine_id: path } a partir de df_machines

    PARA CADA machine_id EM machine_ids:
        GET para base_url/v1/machines/{machine_id}/points

        SE status == 200:
            adicionar registro: { path, MachineId, origem, data: resposta.json() }

    RETORNAR DataFrame com uma linha por machine (data ainda aninhada)
```

### Expansão dos pontos (explode)
```
FUNÇÃO explode_points(df_raw):
    PARA CADA linha do df_raw:
        PARA CADA ponto em linha.data:
            extrair: { origem, path, MachineID, SubmachineID, ID,
                       Name, Description, NodeTypeName, EU, DetectionName }
    RETORNAR DataFrame flat

df_points = explode_points(df_points)
point_ids = lista de IDs dos pontos

SALVAR Excel "cda_online_skf_points.xlsx"
ENVIAR para Google Sheets "cda_online_skf_points"
```

---

## 7. ALARMS

### Busca por machine ID (extrai alarmes dos pontos)
```
FUNÇÃO get_alarms(token, base_url, machine_ids, origem):
    PARA CADA machine_id EM machine_ids:
        GET para base_url/v1/machines/{machine_id}/points

        SE status != 200: pular

        PARA CADA ponto na resposta:
            registro = { origem, MachineId, ID, HighAlarm=None,
                         HighWarning=None, Freq_AlarmLevel=None, Freq_WarningLevel=None }

            // Extrair OverallAlarm
            summary = ponto.OverallAlarm.Summary
            SE summary não vazio:
                DIVIDIR summary por "/"
                PARA CADA parte:
                    SE "high alarm" → extrair valor → HighAlarm
                    SE "high warning" → extrair valor → HighWarning

            // Extrair frequência Overall
            freq_overall = primeiro item de Frequencies onde Frequency == "Overall"
            extrair Freq_AlarmLevel e Freq_WarningLevel como float

            adicionar registro

    converter colunas numéricas
    RETORNAR DataFrame

PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    machine_ids_origem = IDs filtrados por origem
    df_temp = get_alarms(...)
    acumular

df_alarms = concatenar

SALVAR Excel "cda_online_skf_alarms.xlsx"
ENVIAR para Google Sheets "cda_online_skf_alarms"
```

---

## 8. LAST MEASUREMENTS

### Busca (inclui última medição por ponto)
```
FUNÇÃO get_measurements(token, base_url, machine_ids, origem):
    PARA CADA machine_id EM machine_ids:
        GET para base_url/v1/machines/{machine_id}/points?IncludeLastMeasurement=true

        SE status != 200: pular

        PARA CADA ponto na resposta:
            last = ponto.LastMeasurement
            measurements = last.Measurements (ou [None] se vazio)

            PARA CADA medição em measurements:
                adicionar registro:
                    { ReadingTimeUTC, PointID, Speed, SpeedUnits,
                      Direction, ChannelName, Level, Units, origem }

    RETORNAR DataFrame

PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    machine_ids_origem = IDs filtrados por origem
    df_temp = get_measurements(...)
    acumular

df_lastmeasurements = concatenar

converter ReadingTimeUTC para datetime
FILTRAR: remover linhas com ReadingTimeUTC inválido (NaT)

SALVAR Excel "cda_online_skf_lastmeasurements.xlsx"
ENVIAR para Google Sheets "cda_online_skf_lastmeasurements"
```

---

## 9. MEASUREMENTS (Trend — por período D-1)

### Busca por point ID
```
FUNÇÃO consultar_trends(point_ids, token, base_url, origem):
    start = ontem 00:00:00 UTC
    end   = ontem 23:59:59 UTC
    params = { fromDateUTC: start, toDateUTC: end }

    PARA CADA point_id EM point_ids:
        GET para base_url/v1/points/{point_id}/trendMeasurements com params

        SE status == 200:
            PARA CADA record na resposta:
                base = { ReadingTimeUTC, PointID, origem }
                PARA CADA medição em record.Measurements:
                    adicionar { base + ChannelName, Direction, Level, Units }

    RETORNAR DataFrame

PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    point_ids_origem = IDs de df_points filtrados por origem
    df = consultar_trends(point_ids_origem, ...)
    acumular

df_trends = concatenar
```

### Filtro e carga incremental
```
FILTRAR df_trends:
    manter apenas linhas onde ChannelName EM ['Valor global', 'Overall']
    E Direction == "X"

df_trendMeasurements = selecionar [ReadingTimeUTC, PointID, Level, origem]
                        remover duplicatas

SALVAR Excel "cda_online_skf_measurements.xlsx"

// Carga incremental no Sheets (sem duplicar):
df_existente = ler dados atuais do Sheets
SE aba vazia ou sem colunas-chave:
    inicializar aba com cabeçalho
df_novos = df_trendMeasurements - df_existente (apenas registros novos)
SE df_novos não vazio:
    INSERIR df_novos abaixo da última linha existente
SENÃO:
    IMPRIMIR "Nenhuma medição nova"

// Tratamento de duplicatas (limpeza pós-carga):
ler aba completa
remover duplicatas por [ReadingTimeUTC, PointID, Level, Units]
limpar aba e reescrever dados limpos
```

---

## 10. NOTES

### Busca por origem
```
FUNÇÃO get_notes(token, base_url, origem):
    GET para base_url/v1/notes com Bearer token
    SE erro: lançar exceção

    PARA CADA note na resposta:
        extrair: { NoteID, PointID, NodeName, CreatedAt,
                   Title, Author, Text, Priority, origem }

    RETORNAR DataFrame
```

### Estrutura e organização
```
PARA CADA (origem, base_url) EM BASE_URLS:
    token = obter_token(base_url)
    df = get_notes(token, base_url, origem)
    acumular

df_notes = concatenar

JUNTAR df_notes com df_points pelo PointID → trazer coluna "path"
converter "path" para string
DIVIDIR "path" por "\" → colunas path_0, path_1, ...
normalizar path_1
FILTRAR por origem (mesmo critério das demais entidades)

selecionar colunas [NoteID, path, PointID, NodeName, Author,
                    CreatedAt, Title, Text, Priority, origem]
remover duplicatas

SALVAR Excel "cda_online_skf_notes.xlsx"
ENVIAR para Google Sheets "cda_online_skf_notes" (limpar + escrever)
```

---

## FLUXO GERAL (visão macro)

```
INÍCIO
│
├─ Configurar credenciais e URLs por origem (ubu / germano)
├─ Autenticar no Google Sheets
│
├─ [3]  Buscar MACHINES
│        └─ autenticar por origem → GET /machines
│        └─ filtrar por path, montar df_machines
│        └─ Excel + Sheets
│
├─ [4]  Buscar SUBMACHINES
│        └─ buscar hierarquia completa → percorrer árvore recursivamente
│        └─ filtrar por path, montar df_submachines
│        └─ Excel + Sheets
│
├─ [5]  Buscar MACHINE PARTS
│        └─ GET /machines/{id}/parts → expandir faultFrequencies
│        └─ juntar com df_machines → filtrar por path
│        └─ Excel + Sheets
│
├─ [6]  Buscar POINTS
│        └─ GET /machines/{id}/points → expandir pontos (explode)
│        └─ montar df_points com IDs para uso posterior
│        └─ Excel + Sheets
│
├─ [7]  Buscar ALARMS
│        └─ GET /machines/{id}/points → extrair OverallAlarm + Frequencies
│        └─ parsear texto do summary para valores numéricos
│        └─ Excel + Sheets
│
├─ [8]  Buscar LAST MEASUREMENTS
│        └─ GET /machines/{id}/points?IncludeLastMeasurement=true
│        └─ extrair última medição por ponto
│        └─ Excel + Sheets
│
├─ [9]  Buscar MEASUREMENTS (Trend D-1)
│        └─ GET /points/{id}/trendMeasurements com range de datas (ontem)
│        └─ filtrar: ChannelName "Overall" + Direction "X"
│        └─ carga incremental no Sheets (apenas registros novos)
│        └─ limpeza de duplicatas na aba
│
└─ [10] Buscar NOTES
         └─ GET /notes por origem
         └─ juntar com df_points → filtrar por path
         └─ Excel + Sheets

FIM
```

---

## PADRÃO DE FILTRAGEM POR ORIGEM (comum a todas as pesquisas)

```
// Aplicado após qualquer busca que retorne dados hierárquicos com "path"
1. converter coluna "path" para string
2. DIVIDIR "path" por "\" → gerar colunas path_0, path_1, path_2, ...
3. normalizar path_1: strip + upper
4. FILTRAR: manter apenas linhas onde
       path_1 == mapa[origem]
       mapa = { "ubu": "UBU", "germano": "GERMANO" }
```